# 01 – Recolección, limpieza y EDA de noticias

**Tarea 1 – Noticias peruanas: del titular al clasificador** · Diplomado QLAB

| Campo | Valor |
|---|---|
| Estudiante | Ricardo Antonio Macedo Villegas |
| Opción elegida | **A – La República** (350–400 noticias, ~90–100 por categoría) |
| Fecha de descarga | *se imprime en la celda de configuración (`FECHA_DESCARGA`); cópiala aquí:* `AAAA-MM-DD` |
| URL base | https://larepublica.pe |
| RSS oficiales | https://larepublica.pe/rss/politica.xml · /rss/economia.xml · /rss/deportes.xml · /rss/ciencia.xml · /rss/home.xml |
| Sitemaps de noticias | https://larepublica.pe/sitemap/{seccion}.xml · https://larepublica.pe/sitemap/web/{seccion}.xml |
| Portadas de sección | https://larepublica.pe/politica · /economia · /deportes · /ciencia · /tecnologia |
| robots.txt | https://larepublica.pe/robots.txt |

### Estrategia de recolección (y respeto al medio)

1. **Se lee `robots.txt`** y toda URL pasa por `permitido()` antes de pedirse. La República prohíbe, entre otras, las rutas de paginación (`/pag/`, `/page/`), etiquetas (`/tag/`), buscador y archivo; **no se usan**.
2. **Fuente principal: RSS oficiales** de cada sección. Traen titular, bajada (`<description>`), fecha y URL.
3. Como cada RSS solo trae las noticias más recientes (no alcanza ~100 por categoría), se complementa con los **sitemaps de noticias** que el propio robots.txt publica y con las **portadas de sección**.
4. Para las URLs que no vienen del RSS se descarga la página de la noticia **una sola vez** y se leen sus metadatos (`og:title`, `og:description` = bajada, `article:published_time`). En esas páginas también se descubren enlaces a otras noticias de la misma sección.
5. Buenas prácticas: `User-Agent` descriptivo, **pausa de 1.5 s** entre peticiones, **máximo 1 reintento**, tope de peticiones y **caché en `datos/raw/`**: si el notebook se vuelve a ejecutar, reutiliza lo descargado en lugar de volver a pedirlo.

In [ ]:
import json, re, time
from collections import deque
from datetime import datetime, timezone, timedelta
from email.utils import parsedate_to_datetime
from html import unescape
from pathlib import Path
from urllib import robotparser
from urllib.parse import urljoin, urlparse

import pandas as pd
import plotly.express as px
import plotly.io as pio
import requests
from bs4 import BeautifulSoup, MarkupResemblesLocatorWarning
import warnings
warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)

# Los gráficos se ven interactivos en VS Code y, gracias a kaleido, también como imagen en GitHub
try:
    import kaleido  # noqa: F401
    pio.renderers.default = "plotly_mimetype+png"
except ImportError:
    print("Aviso: instala kaleido (pip install kaleido) para que los gráficos también se vean en GitHub.")

LIMA = timezone(timedelta(hours=-5))
FECHA_DESCARGA = datetime.now(LIMA).strftime("%Y-%m-%d %H:%M")

OPCION = "A"
MEDIO = "La Republica"
BASE = "https://larepublica.pe"

RSS_URLS = [f"{BASE}/rss/{s}.xml" for s in ["politica", "economia", "deportes", "ciencia", "home"]]
SITEMAP_URLS = ([f"{BASE}/sitemap/{s}.xml" for s in ["politica", "economia", "deportes", "ciencia"]]
                + [f"{BASE}/sitemap/web/{s}.xml" for s in ["politica", "economia", "deportes", "ciencia"]])
PORTADAS = [f"{BASE}/{s}" for s in ["politica", "economia", "deportes", "ciencia", "tecnologia"]]

# User-Agent descriptivo (puedes añadir tu correo de contacto si quieres)
USER_AGENT = "QLAB-Diplomado-Tarea1/1.0 (proyecto academico: clasificador de titulares)"
HEADERS = {"User-Agent": USER_AGENT, "Accept-Language": "es-PE,es;q=0.9"}
PAUSA_SEG = 1.5              # pausa entre peticiones (1-2 s)
MAX_REINTENTOS = 1           # máximo 1 reintento
OBJETIVO_POR_CATEGORIA = 100 # meta de noticias por categoría
MAX_PETICIONES = 650         # tope de seguridad de páginas de noticia
RECARGAR = False             # True = ignorar la caché de datos/raw y descargar de nuevo

CATEGORIAS = ["politica", "economia", "deportes", "ciencia_tecnologia"]
COLUMNAS = ["titular", "bajada", "medio", "seccion_original", "url", "fecha", "etiqueta"]

DATOS = Path("datos")
RAW = DATOS / "raw"
RAW.mkdir(parents=True, exist_ok=True)

print("Fecha de descarga :", FECHA_DESCARGA)
print("Opción / medio    :", OPCION, "/", MEDIO)
print("RSS               :", *RSS_URLS, sep="\n  ")

## 1. robots.txt y funciones de descarga

`permitido(url)` combina el lector oficial de Python (`urllib.robotparser`) con una lista explícita de las rutas que La República prohíbe, porque `robotparser` no entiende los comodines `*` que usa este robots.txt.

In [ ]:
SESION = requests.Session()
SESION.headers.update(HEADERS)

r = SESION.get(f"{BASE}/robots.txt", timeout=20)
(RAW / "robots.txt").write_text(r.text, encoding="utf-8")
rp = robotparser.RobotFileParser()
rp.parse(r.text.splitlines())

# Rutas prohibidas por el robots.txt de La República (escritas con comodines en el archivo)
PROHIBIDAS = [r"/pag/", r"/page/", r"/tag/", r"^/buscador", r"^/archivo", r"^/archive", r"^/envivo",
              r"^/data", r"^/node/", r"^/taxonomy/", r"^/assets/", r"^/files/", r"/pf/", r"/sitemaps/",
              r"/encuesta", r"/empresa/"]

def permitido(url):
    ruta = urlparse(url).path
    if any(re.search(p, ruta, flags=re.I) for p in PROHIBIDAS):
        return False
    return rp.can_fetch(USER_AGENT, url)

contador = {"peticiones": 0, "fallidas": 0, "bloqueadas_robots": 0}

def descargar(url):
    # GET educado: respeta robots.txt, pausa antes de cada petición y máximo 1 reintento
    if not permitido(url):
        contador["bloqueadas_robots"] += 1
        return None
    for intento in range(MAX_REINTENTOS + 1):
        time.sleep(PAUSA_SEG)
        contador["peticiones"] += 1
        try:
            resp = SESION.get(url, timeout=20)
            if resp.status_code == 200:
                resp.encoding = resp.encoding or "utf-8"
                return resp.text
            print(f"  HTTP {resp.status_code}: {url}")
            if resp.status_code in (401, 403, 404, 410):
                break  # no tiene sentido reintentar
        except requests.RequestException as e:
            print(f"  {type(e).__name__}: {url}")
    contador["fallidas"] += 1
    return None

def descargar_con_cache(url, nombre):
    # Guarda el XML/HTML crudo en datos/raw/ y lo reutiliza en ejecuciones siguientes
    ruta = RAW / nombre
    if ruta.exists() and not RECARGAR:
        return ruta.read_text(encoding="utf-8")
    texto = descargar(url)
    if texto:
        ruta.write_text(texto, encoding="utf-8")
    return texto

for prueba in [f"{BASE}/rss/politica.xml", f"{BASE}/politica/2026/10/18/ejemplo-hnews-1",
               f"{BASE}/politica/pag/2", f"{BASE}/tag/keiko-fujimori"]:
    print(f"{'PERMITIDA ' if permitido(prueba) else 'PROHIBIDA '} {prueba}")

## 2. Funciones para leer RSS, sitemaps, portadas y páginas de noticia

In [ ]:
# Una URL de noticia de La República tiene la forma /seccion[/subseccion]/AAAA/MM/DD/slug
PATRON_NOTICIA = re.compile(r"^https?://larepublica\.pe/((?:[a-z0-9-]+/)+?)(\d{4})/(\d{2})/(\d{2})/[^/?#]+$")

def normalizar_url(url):
    return (url or "").strip().split("#")[0].split("?")[0]

def info_url(url):
    m = PATRON_NOTICIA.match(normalizar_url(url))
    if not m:
        return None
    return {"seccion_original": m.group(1).strip("/"), "fecha_url": f"{m.group(2)}-{m.group(3)}-{m.group(4)}"}

def limpiar(texto):
    if not texto:
        return ""
    texto = BeautifulSoup(unescape(texto), "html.parser").get_text(" ")
    return re.sub(r"\s+", " ", texto).strip()

def a_fecha(valor):
    # Convierte fechas RSS (RFC 822) o ISO a AAAA-MM-DD en hora de Lima
    if not valor:
        return ""
    valor = valor.strip()
    try:
        return parsedate_to_datetime(valor).astimezone(LIMA).strftime("%Y-%m-%d")
    except (TypeError, ValueError):
        pass
    try:
        return datetime.fromisoformat(valor.replace("Z", "+00:00")).astimezone(LIMA).strftime("%Y-%m-%d")
    except ValueError:
        return valor[:10]

def parse_rss(xml, fuente):
    soup = BeautifulSoup(xml, "xml")
    filas = []
    for it in soup.find_all("item"):
        url = normalizar_url(it.find("link").get_text() if it.find("link") else "")
        info = info_url(url) or {}
        cat = it.find("category")
        filas.append({
            "titular": limpiar(it.find("title").get_text() if it.find("title") else ""),
            "bajada": limpiar(it.find("description").get_text() if it.find("description") else ""),
            "url": url,
            "fecha": a_fecha(it.find("pubDate").get_text() if it.find("pubDate") else "") or info.get("fecha_url", ""),
            "seccion_original": info.get("seccion_original", ""),
            "categoria_rss": cat.get_text(strip=True) if cat else "",
            "fuente": fuente,
        })
    return filas

def parse_sitemap(xml, fuente):
    soup = BeautifulSoup(xml, "xml")
    filas = []
    for u in soup.find_all("url"):
        url = normalizar_url(u.find("loc").get_text() if u.find("loc") else "")
        info = info_url(url)
        if not info:
            continue
        titulo = u.find(lambda t: t.name == "title" and t.prefix == "news")
        fecha = u.find("publication_date") or u.find("lastmod")
        filas.append({
            "titular": limpiar(titulo.get_text()) if titulo else "",
            "bajada": "",  # los sitemaps no traen bajada: se obtiene de la página
            "url": url,
            "fecha": info["fecha_url"] or a_fecha(fecha.get_text() if fecha else ""),
            "seccion_original": info["seccion_original"],
            "categoria_rss": "",
            "fuente": fuente,
        })
    return filas

def enlaces_de_noticias(html):
    soup = BeautifulSoup(html, "lxml")
    urls = {normalizar_url(urljoin(BASE, a["href"])) for a in soup.find_all("a", href=True)}
    return sorted(u for u in urls if info_url(u))

def parse_articulo(html, url):
    soup = BeautifulSoup(html, "lxml")
    def meta(*claves):
        for k in claves:
            tag = soup.find("meta", attrs={"property": k}) or soup.find("meta", attrs={"name": k})
            if tag and tag.get("content"):
                return limpiar(tag["content"])
        return ""
    titular = meta("og:title", "twitter:title") or (limpiar(soup.h1.get_text()) if soup.h1 else "")
    titular = re.sub(r"\s*[|\-–]\s*La Rep[uú]blica.*$", "", titular, flags=re.I)
    info = info_url(url) or {}
    datos = {
        "titular": titular,
        "bajada": meta("og:description", "description", "twitter:description"),
        "url": url,
        "fecha": a_fecha(meta("article:published_time")) or info.get("fecha_url", ""),
        "seccion_original": info.get("seccion_original", ""),
        "categoria_rss": meta("article:section"),
        "fuente": "pagina_noticia",
    }
    return datos, enlaces_de_noticias(html)

print("Funciones listas. Ejemplo:", info_url("https://larepublica.pe/deportes/futbol-peruano/2026/10/18/ejemplo-123"))

## 3. Tabla de mapeo sección → categoría

La sección se toma de la propia URL (lo que el diario usa para organizar su web). Se mapea según el **primer nivel** de la sección; cualquier otra sección (mundo, sociedad, espectáculos, opinión, etc.) **se descarta**, sin crear una quinta categoría.

In [ ]:
MAPEO_SECCIONES = {
    "politica":   "politica",
    "economia":   "economia",
    "deportes":   "deportes",
    "ciencia":    "ciencia_tecnologia",
    "tecnologia": "ciencia_tecnologia",
}

def mapear(seccion_original):
    if not isinstance(seccion_original, str) or not seccion_original:
        return None
    return MAPEO_SECCIONES.get(seccion_original.split("/")[0])

pd.DataFrame([{"seccion_original (primer nivel)": k, "etiqueta": v} for k, v in MAPEO_SECCIONES.items()]
             + [{"seccion_original (primer nivel)": "cualquier otra (mundo, sociedad, espectaculos, opinion...)",
                 "etiqueta": "DESCARTADA"}])

## 4. Extracción

**Paso A:** RSS + sitemaps + portadas (pocas peticiones). **Paso B:** se visitan páginas de noticia de las categorías que aún no llegan a la meta, priorizando la categoría con menos noticias. Con 1.5 s de pausa, esta celda puede tardar **10–15 minutos** la primera vez; las siguientes veces usa la caché.

In [ ]:
registros = {}      # url -> datos de la noticia
otras_secciones = []  # noticias de secciones fuera de las 4 categorías (para el reporte)

def agregar(fila):
    url = fila["url"]
    if not url:
        return
    if url in registros:
        actual = registros[url]
        for k, v in fila.items():   # completa campos vacíos sin pisar los existentes
            if v and not actual.get(k):
                actual[k] = v
    else:
        registros[url] = dict(fila)

# --- Paso A.1: RSS oficiales
for url in RSS_URLS:
    xml = descargar_con_cache(url, "rss_" + url.rsplit("/", 1)[-1])
    if xml:
        filas = parse_rss(xml, "rss:" + url.rsplit("/", 1)[-1])
        print(f"RSS     {url:<45} {len(filas):>4} noticias")
        for f in filas:
            agregar(f)

# --- Paso A.2: sitemaps de noticias
for url in SITEMAP_URLS:
    nombre = "sitemap_" + urlparse(url).path.strip("/").replace("/", "_")
    xml = descargar_con_cache(url, nombre)
    if xml:
        filas = parse_sitemap(xml, "sitemap")
        print(f"Sitemap {url:<45} {len(filas):>4} noticias")
        for f in filas:
            agregar(f)

# --- Paso A.3: portadas de sección (solo la primera página; la paginación está prohibida)
for url in PORTADAS:
    html = descargar_con_cache(url, "portada_" + url.rsplit("/", 1)[-1] + ".html")
    if html:
        enlaces = enlaces_de_noticias(html)
        print(f"Portada {url:<45} {len(enlaces):>4} enlaces")
        for e in enlaces:
            agregar({"url": e, "fuente": "portada", **info_url(e)})

def completa(d):
    return bool(d.get("titular")) and bool(d.get("bajada"))

def conteo_completas():
    c = {cat: 0 for cat in CATEGORIAS}
    for d in registros.values():
        cat = mapear(d.get("seccion_original"))
        if cat and completa(d):
            c[cat] += 1
    return c

print("\nNoticias completas tras el paso A:", conteo_completas())

In [ ]:
# --- Paso B: visitar páginas de noticia (con caché en datos/raw/articulos_meta.jsonl)
ruta_cache = RAW / "articulos_meta.jsonl"
cache = {}
if ruta_cache.exists() and not RECARGAR:
    for linea in ruta_cache.read_text(encoding="utf-8").splitlines():
        if linea.strip():
            d = json.loads(linea)
            cache[d["url"]] = d

visitadas = set()
pendientes = {cat: deque() for cat in CATEGORIAS}   # noticias sin bajada/titular
semillas = {cat: deque() for cat in CATEGORIAS}     # noticias completas: solo sirven para descubrir enlaces

for url, d in list(registros.items()):
    cat = mapear(d.get("seccion_original"))
    if cat:
        (semillas if completa(d) else pendientes)[cat].append(url)

def encolar(enlaces):
    for e in enlaces:
        if e in registros:
            continue
        info = info_url(e)
        agregar({"url": e, "fuente": "enlace_descubierto", **info})
        cat = mapear(info["seccion_original"])
        if cat:
            pendientes[cat].append(e)

peticiones_paginas = 0
inicio = time.time()
with open(ruta_cache, "a", encoding="utf-8") as f_cache:
    while peticiones_paginas < MAX_PETICIONES:
        conteo = conteo_completas()
        faltan = [c for c in CATEGORIAS if conteo[c] < OBJETIVO_POR_CATEGORIA and (pendientes[c] or semillas[c])]
        if not faltan:
            break
        cat = min(faltan, key=lambda c: conteo[c])
        cola = pendientes[cat] if pendientes[cat] else semillas[cat]
        url = cola.popleft()
        if url in visitadas:
            continue
        visitadas.add(url)

        if url in cache:
            datos = {k: v for k, v in cache[url].items() if k != "enlaces"}
            enlaces = cache[url].get("enlaces", [])
        else:
            html = descargar(url)
            peticiones_paginas += 1
            if not html:
                continue
            datos, enlaces = parse_articulo(html, url)
            f_cache.write(json.dumps({**datos, "enlaces": enlaces}, ensure_ascii=False) + "\n")
            f_cache.flush()
            if peticiones_paginas % 25 == 0:
                print(f"  {peticiones_paginas} páginas · {time.time() - inicio:.0f} s · {conteo_completas()}")
        agregar(datos)
        encolar(enlaces)

print(f"\nPáginas descargadas en esta ejecución: {peticiones_paginas}")
print("Contador de peticiones:", contador)
print("Noticias completas por categoría:", conteo_completas())

df_crudo = pd.DataFrame(registros.values())
df_crudo.to_csv(RAW / "noticias_crudas.csv", index=False, encoding="utf-8")
print("Registros crudos (incluye incompletos y otras secciones):", df_crudo.shape)

## 5. Limpieza y normalización

Orden: normalizar URL → quitar duplicados por URL → quitar titulares vacíos → quitar noticias sin bajada → mapear sección → quitar secciones fuera de las 4 categorías → quitar titulares repetidos → balancear a un máximo de 100 por categoría (se quedan las más recientes). Cada descarte queda registrado con su motivo.

In [ ]:
df = df_crudo.copy()
for col in ["titular", "bajada", "url", "fecha", "seccion_original"]:
    df[col] = df[col].fillna("").astype(str).str.strip()
df["url"] = df["url"].map(normalizar_url)
df["medio"] = MEDIO

# Los enlaces descubiertos que nunca se visitaron (porque ya se alcanzó la meta) no son noticias recolectadas
no_visitados = df["titular"].eq("") & ~df["url"].isin(visitadas)
print(f"Enlaces descubiertos que no se visitaron (meta ya alcanzada): {no_visitados.sum()} -> no cuentan como noticias")
df = df[~no_visitados].copy()
total_inicial = len(df)

descartes = []
def descartar(mascara, motivo):
    global df
    n = int(mascara.sum())
    if n:
        d = df[mascara].copy()
        d["motivo"] = motivo
        descartes.append(d)
    df = df[~mascara].copy()
    print(f"- {motivo:<55} {n:>5}")

print("Descartes:")
descartar(df["url"].eq("") | df["url"].map(lambda u: info_url(u) is None), "URL vacía o que no es una noticia")
descartar(df.duplicated("url", keep="first"), "URL duplicada")
descartar(df["titular"].eq(""), "Titular vacío (página no disponible)")
descartar(df["bajada"].eq(""), "Bajada vacía")
df["etiqueta"] = df["seccion_original"].map(mapear)
descartar(df["etiqueta"].isna(), "Sección fuera de las 4 categorías")
descartar(df.duplicated("titular", keep="first"), "Titular repetido con otra URL")

df = df.sort_values("fecha", ascending=False)
exceso = df.groupby("etiqueta").cumcount() >= OBJETIVO_POR_CATEGORIA
descartar(exceso, f"Exceso sobre {OBJETIVO_POR_CATEGORIA} por categoría (balanceo)")

df_descartes = pd.concat(descartes, ignore_index=True) if descartes else pd.DataFrame(columns=["motivo"])
print(f"\nNoticias iniciales: {total_inicial} | descartadas: {len(df_descartes)} | finales: {len(df)}")

In [ ]:
# Reporte de descartes por motivo
reporte = df_descartes["motivo"].value_counts().rename_axis("motivo").reset_index(name="noticias")
reporte.loc[len(reporte)] = ["TOTAL descartadas", reporte["noticias"].sum()]
reporte

In [ ]:
# Ejemplos de noticias descartadas por estar fuera de las 4 categorías
fuera = df_descartes[df_descartes["motivo"] == "Sección fuera de las 4 categorías"]
print("Secciones descartadas:", fuera["seccion_original"].str.split("/").str[0].value_counts().to_dict())
fuera[["titular", "seccion_original", "url"]].head(10)

In [ ]:
# Tabla de equivalencias REAL observada: sección original -> categoría final
tabla_mapeo = (df.groupby(["seccion_original", "etiqueta"]).size()
                 .reset_index(name="noticias").sort_values(["etiqueta", "noticias"], ascending=[True, False]))
tabla_mapeo

## 6. Análisis exploratorio (EDA)

Los títulos de los gráficos se calculan con los datos, para que expresen la conclusión y no solo la variable.

In [ ]:
df["largo_titular"] = df["titular"].str.len()
df["fecha_dt"] = pd.to_datetime(df["fecha"], errors="coerce")

# Gráfico 1: distribución por categoría
conteo = df["etiqueta"].value_counts().reindex(CATEGORIAS).reset_index()
conteo.columns = ["etiqueta", "noticias"]
mayor = conteo.loc[conteo["noticias"].idxmax()]
menor = conteo.loc[conteo["noticias"].idxmin()]
if mayor["noticias"] == menor["noticias"]:
    titulo1 = f"Dataset perfectamente balanceado: {mayor['noticias']} noticias en cada una de las 4 categorías"
else:
    titulo1 = (f"Dataset casi balanceado: de {menor['noticias']} ({menor['etiqueta']}) "
               f"a {mayor['noticias']} ({mayor['etiqueta']}) noticias por categoría")
fig1 = px.bar(conteo, x="etiqueta", y="noticias", color="etiqueta", text="noticias", title=titulo1,
              labels={"etiqueta": "Categoría", "noticias": "Número de noticias"})
fig1.update_layout(showlegend=False)
fig1.show()

In [ ]:
# Gráfico 2: distribución por rango de fechas
semanal = (df.dropna(subset=["fecha_dt"])
             .assign(semana=lambda d: d["fecha_dt"].dt.to_period("W").dt.start_time)
             .groupby(["semana", "etiqueta"]).size().reset_index(name="noticias"))
ultimos7 = (df["fecha_dt"] >= df["fecha_dt"].max() - pd.Timedelta(days=6)).mean() * 100
rango = f"{df['fecha_dt'].min():%d/%m/%Y} – {df['fecha_dt'].max():%d/%m/%Y}"
if ultimos7 >= 50:
    titulo2 = f"El {ultimos7:.0f}% de las noticias es de la última semana ({rango}): el corpus refleja la coyuntura reciente"
else:
    titulo2 = f"Solo el {ultimos7:.0f}% es de la última semana: el corpus cubre varias semanas ({rango})"
fig2 = px.bar(semanal, x="semana", y="noticias", color="etiqueta", title=titulo2,
              labels={"semana": "Semana de publicación", "noticias": "Noticias", "etiqueta": "Categoría"})
fig2.show()

In [ ]:
# Gráfico 3: histograma de longitud de los titulares
mediana = df["largo_titular"].median()
por_cat = df.groupby("etiqueta")["largo_titular"].median().sort_values()
titulo3 = (f"Los titulares miden {mediana:.0f} caracteres en mediana; {por_cat.index[-1]} "
           f"titula más largo ({por_cat.iloc[-1]:.0f}) y {por_cat.index[0]} más corto ({por_cat.iloc[0]:.0f})")
fig3 = px.histogram(df, x="largo_titular", color="etiqueta", nbins=30, marginal="box", barmode="overlay",
                    opacity=0.65, title=titulo3,
                    labels={"largo_titular": "Longitud del titular (caracteres)", "etiqueta": "Categoría"})
fig3.update_layout(yaxis_title="Número de titulares")
fig3.show()
print(df.groupby("etiqueta")["largo_titular"].describe().round(1))

## 7. Validación: titulares más largos y más cortos

Si el histograma está bien construido, los valores extremos de esta tabla deben coincidir con los bordes del histograma (mínimo y máximo).

In [ ]:
pd.set_option("display.max_colwidth", 140)
print("5 titulares MÁS LARGOS")
display(df.nlargest(5, "largo_titular")[["largo_titular", "etiqueta", "titular"]])
print("5 titulares MÁS CORTOS")
display(df.nsmallest(5, "largo_titular")[["largo_titular", "etiqueta", "titular"]])
print(f"Coherencia con el histograma -> mínimo: {df['largo_titular'].min()} | "
      f"máximo: {df['largo_titular'].max()} | percentil 95: {df['largo_titular'].quantile(.95):.0f}")

## 8. Verificación de URLs funcionales y exportación

In [ ]:
# Se comprueban 10 URLs al azar (con pausa) para demostrar que los enlaces funcionan
muestra_urls = df.sample(min(10, len(df)), random_state=42)["url"]
estados = []
for u in muestra_urls:
    time.sleep(PAUSA_SEG)
    try:
        estados.append({"url": u, "status": SESION.get(u, timeout=20, stream=True).status_code})
    except requests.RequestException as e:
        estados.append({"url": u, "status": type(e).__name__})
verif = pd.DataFrame(estados)
print(f"URLs con respuesta 200: {(verif['status'] == 200).sum()}/{len(verif)}")
verif

In [ ]:
noticias = df[COLUMNAS].reset_index(drop=True)
noticias.to_csv(DATOS / "noticias.csv", index=False, encoding="utf-8")
print("Guardado en datos/noticias.csv")
print("df.shape =", noticias.shape)
assert noticias.shape[1] == 7, "El CSV debe tener exactamente 7 columnas"
if len(noticias) < 350:
    print(f"ATENCIÓN: solo hay {len(noticias)} filas (< 350). Sube MAX_PETICIONES y vuelve a ejecutar.")
print(noticias["etiqueta"].value_counts())
noticias.head(5)